# Bilge Pump System — SysML v2 Verification Notebook

## Overview
This notebook verifies the `BilgePumpVerification` analysis case against the four-layer SysML v2 model.

| Layer | File | Contents |
|---|---|---|
| Library | `Library.sysml` | All `part def`, `port def`, `attribute def` |
| Architecture | `Architecture.sysml` | 8 part usages + 11 connect statements |
| Requirements | `Requirements.sysml` | 4 `requirement def` blocks (BPS-REQ-001 through 004) |
| Analysis | `Analysis.sysml` | Physics `constraint def` + `analysis def` test runner |

## API Server
This notebook uses the **SST public SysML v2 API** — no local install required.

| Item | Value |
|---|---|
| Endpoint | `http://sysml2.intercax.com:9000` |
| Managed by | SysML Submission Team (public service) |
| Auth | None |
| Note | Projects are visible to all users of the shared server |

## Prerequisites
```bash
bash setup.sh   # one-time: installs Python deps + confirms API reachability
```
Then open this notebook (`bash run.sh` or directly in VS Code).

## How the Analysis Case works as a Test Runner
The `analysis def BilgePumpVerification` in `Analysis.sysml` is the integration harness:
1. **Subject binding** — `bind` statements wire attribute values to the system instance
2. **Test cases** — each `assert requirement` is one test; constraint body is evaluated
3. **Physics check** — `PumpFlowPhysics` constraint validates Q_net ≥ design inflow
4. **Results** — SATISFIED / VIOLATED per requirement

## How verification works in this notebook
The SST REST API is a **model element store** for IDE tooling (VS Code SysML extension, Eclipse SysIDE).
It stores the SysML text verbatim but does not execute it as a constraint solver.
Cells 7 and 8 therefore evaluate the `require constraint` expressions **locally in Python**,
using the same logic as `bash verify.sh`. The API is still used (Cells 2–6) to persist the model.

Run cells **top to bottom**. Cell 7 is the main pass/fail report. Cell 8 is the negative test.

In [1]:
# =============================================================================
# Cell 2: Connect to the SST public SysML v2 API server and create a project
#
# API endpoint: http://sysml2.intercax.com:9000
# This is the SST (SysML Submission Team) publicly hosted server.
# No local install or JAR needed.
# =============================================================================
import requests
import json
import time
import os

API_BASE = "http://sysml2.intercax.com:9000"
SCRIPT_DIR = os.path.dirname(os.path.abspath("Verification.ipynb"))

# --- Health check ---
print(f"Connecting to SysML v2 API server at {API_BASE}...")
try:
    r = requests.get(f"{API_BASE}/projects", timeout=8)
    r.raise_for_status()
    print(f"  Server ready. Existing projects on server: {len(r.json())}")
except Exception as e:
    raise RuntimeError(
        f"Cannot reach {API_BASE}/projects — check internet/firewall.\nError: {e}"
    )

# --- Create a new project ---
project_payload = {"name": "BilgePumpSystem", "description": "SysML v2 bilge pump verification project"}
r = requests.post(f"{API_BASE}/projects", json=project_payload)
r.raise_for_status()
project = r.json()
PROJECT_ID = project["@id"]

print(f"\nProject created:")
print(f"  ID   : {PROJECT_ID}")
print(f"  Name : {project.get('name', 'BilgePumpSystem')}")
print(f"  Note : This project is on a shared public server.")


Connecting to SysML v2 API server at http://sysml2.intercax.com:9000...
  Server ready. Existing projects on server: 100

Project created:
  ID   : 79395c31-36cc-4d07-96e3-7f055af7bdbf
  Name : BilgePumpSystem
  Note : This project is on a shared public server.


In [2]:
# =============================================================================
# Cell 3: Commit Library.sysml
# Loads all part def, port def, attribute def into the project.
# =============================================================================

def commit_sysml_file(project_id: str, filepath: str, description: str) -> dict:
    """Read a .sysml file and POST it as a commit to the API server."""
    with open(filepath, "r") as f:
        content = f.read()

    payload = {
        "description": description,
        "changes": [
            {
                "@type": "TextualRepresentation",
                "body": content
            }
        ]
    }
    r = requests.post(
        f"{API_BASE}/projects/{project_id}/commits",
        json=payload
    )
    r.raise_for_status()
    result = r.json()
    print(f"  Committed: {filepath}  (commit id: {result.get('@id', 'n/a')})")
    return result

print("[1/4] Committing Library.sysml...")
lib_commit = commit_sysml_file(
    PROJECT_ID,
    os.path.join(SCRIPT_DIR, "Library.sysml"),
    "Library layer: part def, port def, attribute def"
)
print("      Done.")

[1/4] Committing Library.sysml...
  Committed: /home/manret/SysMLInfra/Library.sysml  (commit id: 4afdbd56-1d23-46a8-8a7e-698e732f0cfd)
      Done.


In [3]:
# =============================================================================
# Cell 4: Commit Architecture.sysml
# Loads BilgePumpSystem with 8 part usages and 11 connect statements.
# The API server validates port type compatibility at this step.
# =============================================================================
print("[2/4] Committing Architecture.sysml...")
arch_commit = commit_sysml_file(
    PROJECT_ID,
    os.path.join(SCRIPT_DIR, "Architecture.sysml"),
    "Architecture layer: BilgePumpSystem part usages and connections"
)
print("      Done.")

[2/4] Committing Architecture.sysml...
  Committed: /home/manret/SysMLInfra/Architecture.sysml  (commit id: 804b740b-cfdb-46cf-af51-c405d00925b6)
      Done.


In [4]:
# =============================================================================
# Cell 5: Commit Requirements.sysml
# Loads all 4 requirement def blocks.
# =============================================================================
print("[3/4] Committing Requirements.sysml...")
req_commit = commit_sysml_file(
    PROJECT_ID,
    os.path.join(SCRIPT_DIR, "Requirements.sysml"),
    "Requirements layer: 4 requirement defs with assert constraints"
)
print("      Done.")

[3/4] Committing Requirements.sysml...
  Committed: /home/manret/SysMLInfra/Requirements.sysml  (commit id: c729b2ab-46bf-4f85-b009-0bb605e87fbc)
      Done.


In [5]:
# =============================================================================
# Cell 6: Commit Analysis.sysml
# Loads PumpFlowPhysics constraint def and BilgePumpVerification analysis def.
# =============================================================================
print("[4/4] Committing Analysis.sysml...")
ana_commit = commit_sysml_file(
    PROJECT_ID,
    os.path.join(SCRIPT_DIR, "Analysis.sysml"),
    "Analysis layer: PumpFlowPhysics constraint + BilgePumpVerification analysis def"
)
print("      Done.")
print("\nAll 4 layers committed. Model is fully loaded.")

[4/4] Committing Analysis.sysml...
  Committed: /home/manret/SysMLInfra/Analysis.sysml  (commit id: 984052d0-e8f7-4f5d-a2c1-6dc52b537c23)
      Done.

All 4 layers committed. Model is fully loaded.


In [6]:
# =============================================================================
# Cell 7: POSITIVE TEST — Evaluate BilgePumpVerification requirements
#
# The SST REST API stores SysML text but does not execute it as a solver.
# Verification is performed by parsing bind values from Analysis.sysml and
# evaluating each require constraint expression in Python.
# This is the same logic as `bash verify.sh`.
#
# Nominal bind values from Analysis.sysml:
#   sensor.waterLevel        = 0.15 m   (threshold: ≤ 0.30)
#   pumpB.isRedundant        = true
#   alarm.activationDelay_s  = 0.50 s   (threshold: ≤ 2.00)
#   pumpA.flowRate           = 0.025 m³/s
#   pumpB.flowRate           = 0.025 m³/s
#   efficiency               = 0.82
#   pipeLossFactor           = 0.05
#   designInflow             = 0.030 m³/s
#   → Q_net = (0.025+0.025) × 0.82 × (1−0.05) = 0.0389 m³/s  ≥ 0.030 ✓
# =============================================================================

import re

def strip_comments(txt):
    txt = re.sub(r'/\*.*?\*/', '', txt, flags=re.DOTALL)
    txt = re.sub(r'//[^\n]*', '', txt)
    return txt

def parse_bind_values(analysis_text):
    """Extract numeric/boolean bind values from Analysis.sysml."""
    values = {}
    for m in re.finditer(r'bind\s+([\w.]+)\s*=\s*([^;]+);', analysis_text):
        path, raw = m.group(1).strip(), m.group(2).strip()
        if raw.lower() == "true":
            values[path] = True
        elif raw.lower() == "false":
            values[path] = False
        else:
            try:
                values[path] = float(raw)
            except ValueError:
                pass  # skip non-literal binds (bind x = sys.y.z)
    return values

def evaluate_requirements(bind_values, requirements_text, test_label, pumpA_override=None):
    """Parse and evaluate all require constraint blocks against bound values."""
    bare = {k.rsplit('.', 1)[-1]: v for k, v in bind_values.items()}
    if pumpA_override is not None:
        for key in list(bind_values.keys()):
            if "pumpa" in key.lower() and "flowrate" in key.lower():
                bind_values[key] = pumpA_override
                bare[key.rsplit('.', 1)[-1]] = pumpA_override
                print(f"  [OVERRIDE] {key} = {pumpA_override}  (failure simulation)")

    req_labels = {
        "WaterLevelRequirement":        "BPS-REQ-001  Water level ≤ 0.30 m",
        "PumpRedundancyRequirement":    "BPS-REQ-002  Pump B redundancy active",
        "AlarmResponseRequirement":     "BPS-REQ-003  Alarm delay ≤ 2.00 s",
        "DischargeCapacityRequirement": "BPS-REQ-004  Discharge ≥ design inflow",
    }

    req_pattern = re.compile(
        r'requirement\s+def\s+(\w+).*?require\s+constraint\s*\{([^}]+)\}',
        re.DOTALL
    )

    results = []
    all_pass = True
    print(f"\n{'='*62}")
    print(f"  {test_label}")
    print(f"{'='*62}")
    for m in req_pattern.finditer(requirements_text):
        req_name = m.group(1)
        expr = m.group(2).strip()
        for path, val in sorted(bind_values.items(), key=lambda x: -len(x[0])):
            expr = re.sub(r'\b' + re.escape(path) + r'\b', repr(val), expr)
        for b, val in sorted(bare.items(), key=lambda x: -len(x[0])):
            expr = re.sub(r'\b' + re.escape(b) + r'\b', repr(val), expr)
        expr = re.sub(r'\btrue\b', 'True', expr)
        expr = re.sub(r'\bfalse\b', 'False', expr)
        expr_clean = re.sub(r'(?<=[\d)])\s+[a-zA-Z/³²°]+', '', expr).strip()
        try:
            satisfied = bool(eval(expr_clean, {"__builtins__": {}}))
        except Exception as e:
            satisfied = None
            print(f"  EVAL ERROR in {req_name}: {e}  (expr: {expr_clean!r})")
        if satisfied is not True:
            all_pass = False
        status = "SATISFIED ✓" if satisfied is True else ("VIOLATED  ✗" if satisfied is False else "UNKNOWN   ?")
        print(f"  {status}   {req_labels.get(req_name, req_name)}")
        results.append({"requirement": req_name, "satisfied": satisfied})
    print(f"{'='*62}")
    print(f"  Overall: {'ALL SATISFIED ✓' if all_pass else 'FAILURES DETECTED ✗'}")
    print(f"{'='*62}")
    return results, all_pass


# Load and strip comments from source files
analysis_clean      = strip_comments(open(os.path.join(SCRIPT_DIR, "Analysis.sysml")).read())
requirements_clean  = strip_comments(open(os.path.join(SCRIPT_DIR, "Requirements.sysml")).read())

print("Running POSITIVE TEST (nominal values from Analysis.sysml)...")
bind_vals = parse_bind_values(analysis_clean)
positive_results, positive_pass = evaluate_requirements(
    bind_vals, requirements_clean, "POSITIVE TEST — Nominal Values"
)


Running POSITIVE TEST (nominal values from Analysis.sysml)...

  POSITIVE TEST — Nominal Values
  SATISFIED ✓   BPS-REQ-001  Water level ≤ 0.30 m
  SATISFIED ✓   BPS-REQ-002  Pump B redundancy active
  SATISFIED ✓   BPS-REQ-003  Alarm delay ≤ 2.00 s
  SATISFIED ✓   BPS-REQ-004  Discharge ≥ design inflow
  Overall: ALL SATISFIED ✓


In [7]:
# =============================================================================
# Cell 8: NEGATIVE TEST — Inject pump A failure; confirm harness detects it
#
# Overrides pumpA.flowRate = 0.0 in memory (no new commit needed).
#   Combined flow = 0.0 + 0.025 = 0.025 m³/s
#   Q_net = 0.025 × 0.82 × 0.95 = 0.0195 m³/s  <  0.030 m³/s design inflow
#   DischargeCapacityRequirement MUST be VIOLATED.
# Other 3 requirements should remain SATISFIED.
#
# In a FMEA workflow this cell would be parameterised over a fault list.
# =============================================================================

print("Running NEGATIVE TEST (pumpA.flowRate = 0.0 — pump A offline)...")
bind_vals_neg = parse_bind_values(analysis_clean)   # fresh copy
negative_results, negative_pass = evaluate_requirements(
    bind_vals_neg,
    requirements_clean,
    "NEGATIVE TEST — Pump A Failure (flowRate = 0)",
    pumpA_override=0.0
)

print("\nExpected: BPS-REQ-004 (DischargeCapacityRequirement) VIOLATED, all others SATISFIED.")
if not negative_pass:
    print("Harness correctly detected the failure. ✓")
else:
    print("WARNING: failure was NOT detected — check constraint logic.")


Running NEGATIVE TEST (pumpA.flowRate = 0.0 — pump A offline)...
  [OVERRIDE] sys.pumpA.flowRate = 0.0  (failure simulation)

  NEGATIVE TEST — Pump A Failure (flowRate = 0)
  SATISFIED ✓   BPS-REQ-001  Water level ≤ 0.30 m
  SATISFIED ✓   BPS-REQ-002  Pump B redundancy active
  SATISFIED ✓   BPS-REQ-003  Alarm delay ≤ 2.00 s
  VIOLATED  ✗   BPS-REQ-004  Discharge ≥ design inflow
  Overall: FAILURES DETECTED ✗

Expected: BPS-REQ-004 (DischargeCapacityRequirement) VIOLATED, all others SATISFIED.
Harness correctly detected the failure. ✓


In [8]:
# =============================================================================
# Cell 9: FEATURE SHOWCASE — Port connection graph
#
# The SST REST API stores SysML text verbatim as TextualRepresentation objects.
# Individual model elements are not indexed separately by this server build,
# so we derive the connection graph by parsing Architecture.sysml directly.
# In an IDE-integrated workflow (VS Code SysML extension, Eclipse SysIDE),
# these connections would be queryable as first-class JSON-LD elements via
# GET /projects/{id}/commits/{cid}/elements.
# =============================================================================

arch_text = open(os.path.join(SCRIPT_DIR, "Architecture.sysml")).read()

# Parse all 'connect <src> to <dst>;' statements
connect_pattern = re.compile(r'connect\s+([\w.]+)\s+to\s+([\w.]+)\s*;')
connections = connect_pattern.findall(arch_text)

signal_type = {
    "levelOut":      "LevelSignal",
    "pumpAControl":  "ControlSignal",
    "pumpBControl":  "ControlSignal",
    "powerOutA":     "PowerSupply",
    "powerOutB":     "PowerSupply",
    "flowOut":       "FluidFlow",
    "statusOut":     "StatusSignal",
    "overrideOut":   "ControlSignal",
    "alarmOut":      "AlarmSignal",
    "notifyOut":     "AlarmSignal",
}

print(f"{'='*68}")
print(f"  FEATURE SHOWCASE: BilgePumpSystem Port Connection Graph")
print(f"  Parsed from Architecture.sysml — {len(connections)} connections found (expected: 11)")
print(f"{'='*68}")
print(f"  {'Source':<32} {'→':<2} {'Destination':<30} {'Signal type'}")
print(f"  {'-'*64}")
for src, dst in connections:
    port = src.split(".")[-1]
    sig  = signal_type.get(port, "")
    print(f"  {src:<32} →  {dst:<30} {sig}")
print(f"{'='*68}")


  FEATURE SHOWCASE: BilgePumpSystem Port Connection Graph
  Parsed from Architecture.sysml — 11 connections found (expected: 11)
  Source                           →  Destination                    Signal type
  ----------------------------------------------------------------
  sensor.levelOut                  →  controller.levelIn             LevelSignal
  controller.pumpAControl          →  pumpA.controlIn                ControlSignal
  controller.pumpBControl          →  pumpB.controlIn                ControlSignal
  power.powerOutA                  →  pumpA.powerIn                  PowerSupply
  power.powerOutB                  →  pumpB.powerIn                  PowerSupply
  pumpA.flowOut                    →  discharge.flowInA              FluidFlow
  pumpB.flowOut                    →  discharge.flowInB              FluidFlow
  controller.statusOut             →  ui.statusIn                    StatusSignal
  ui.overrideOut                   →  controller.overrideIn          Contr

In [9]:
# =============================================================================
# Cell 10: FEATURE SHOWCASE — PumpFlowPhysics constraint evaluation
#
# Evaluates the physics equation:
#   Q_net = (Q_A + Q_B) × η × (1 − λ)
# with nominal values and shows whether the system meets design inflow.
#
# This cell works independently of the API server — it evaluates the
# constraint in Python to mirror what the SysML v2 engine computes.
# =============================================================================

# --- Nominal values (matching Analysis.sysml bind statements) ---
flow_rate_A     = 0.025   # m³/s  — CFD pump curve at rated speed
flow_rate_B     = 0.025   # m³/s  — identical redundant pump
efficiency      = 0.82    # η     — hydraulic efficiency (vendor test cert)
pipe_loss       = 0.05    # λ     — Darcy-Weisbach from P&ID hydraulic calc
design_inflow   = 0.030   # m³/s  — from Maxsurf damage stability analysis

# --- Physics equation ---
#   PumpFlowPhysics constraint body:
#   netFlow = (flowRateA + flowRateB) * efficiency * (1.0 - pipeLossFactor)
Q_net = (flow_rate_A + flow_rate_B) * efficiency * (1.0 - pipe_loss)

print("="*60)
print("  FEATURE SHOWCASE: PumpFlowPhysics Constraint Evaluation")
print("="*60)
print(f"  Q_A  (Pump A flow rate)       = {flow_rate_A:.4f} m³/s")
print(f"  Q_B  (Pump B flow rate)       = {flow_rate_B:.4f} m³/s")
print(f"  η    (hydraulic efficiency)   = {efficiency:.2f}")
print(f"  λ    (pipe loss factor)       = {pipe_loss:.2f}")
print(f"  Q_design (design inflow)      = {design_inflow:.4f} m³/s")
print()
print(f"  Formula: Q_net = (Q_A + Q_B) × η × (1 − λ)")
print(f"         = ({flow_rate_A} + {flow_rate_B}) × {efficiency} × {1.0 - pipe_loss}")
print(f"         = {Q_net:.6f} m³/s")
print()

margin = Q_net - design_inflow
if Q_net >= design_inflow:
    print(f"  Q_net ({Q_net:.4f}) ≥ Q_design ({design_inflow:.4f})  →  SATISFIED ✓")
    print(f"  Safety margin: +{margin:.4f} m³/s ({margin/design_inflow*100:.1f}% above requirement)")
else:
    print(f"  Q_net ({Q_net:.4f}) < Q_design ({design_inflow:.4f})  →  VIOLATED ✗")
    print(f"  Deficit: {abs(margin):.4f} m³/s")
print("="*60)

  FEATURE SHOWCASE: PumpFlowPhysics Constraint Evaluation
  Q_A  (Pump A flow rate)       = 0.0250 m³/s
  Q_B  (Pump B flow rate)       = 0.0250 m³/s
  η    (hydraulic efficiency)   = 0.82
  λ    (pipe loss factor)       = 0.05
  Q_design (design inflow)      = 0.0300 m³/s

  Formula: Q_net = (Q_A + Q_B) × η × (1 − λ)
         = (0.025 + 0.025) × 0.82 × 0.95
         = 0.038950 m³/s

  Q_net (0.0389) ≥ Q_design (0.0300)  →  SATISFIED ✓
  Safety margin: +0.0089 m³/s (29.8% above requirement)


## Interpretation Guide

### What each cell produces
| Cell | Output |
|---|---|
| 2 | Project created on SST API; server connectivity confirmed |
| 3–6 | Each layer POSTed as a `TextualRepresentation` commit; HTTP 200 = stored |
| 7 | Pass/fail per requirement (POSITIVE TEST) — all 4 should be `SATISFIED ✓` |
| 8 | NEGATIVE TEST — `BPS-REQ-004 DischargeCapacityRequirement` must be `VIOLATED ✗` |
| 9 | All 11 port connections parsed from Architecture.sysml |
| 10 | Physics equation evaluated; safety margin printed |

### Why verification runs locally
The SST REST API (`sysml2.intercax.com:9000`) is the **SysML v2 Pilot model store**.
It accepts commits from IDE tooling (VS Code SysML extension, Eclipse SysIDE) and exposes
model elements as JSON-LD for navigation and collaboration. It does **not** include a
constraint solver or analysis executor in this deployment.

Cells 7 and 8 evaluate the `require constraint` expressions from `Requirements.sysml`
in Python, substituting the `bind` values from `Analysis.sysml`. This matches the
semantics of the SysML v2 engine for simple arithmetic and boolean constraints.

### How to add a new requirement
1. Add a `requirement def` block in `Requirements.sysml` following the existing pattern
2. Add an `assert requirement` line in `Analysis.sysml` inside `BilgePumpVerification`
3. Re-run from Cell 2 (new project) or just Cell 7 if the project ID is still valid

### How to add a new failure scenario
1. Call `evaluate_requirements(bind_vals, requirements_clean, "label", pumpA_override=X)` in a new cell
2. Or modify the override dict for any attribute — this is the FMEA pattern

### CLI equivalents
```bash
bash commit.sh           # upload all 4 layers (equivalent to Cells 3–6)
bash verify.sh           # positive test  (equivalent to Cell 7)
bash verify.sh negative  # negative test  (equivalent to Cell 8)
bash check-requirements-manually.sh  # same evaluation, standalone
```

### CI/CD note
```yaml
# .github/workflows/verify.yml
- run: pip install requests
- run: bash commit.sh
- run: bash verify.sh        # exits 0 on all SATISFIED, 1 on any VIOLATED
```
No Java, no local server. The SST API is the backend.
